# 01 — Exploratory Data Analysis: 2013 US Flight Delays

This notebook explores the `flights.csv` dataset to understand data quality, distributions, and delay patterns.

**Rules (from ADRs):**
- **Label**: `ArrDel15` = arrival delayed ≥ 15 minutes (ADR-0001)
- **Cancelled flights**: excluded from delay analysis (ADR-0002)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 25)

RANDOM_SEED = 42
WEEKDAY_NAMES = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}

## 1. Load and inspect the dataset

In [ ]:
df = pd.read_csv('../data/flights.csv')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Missing values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

## 3. Filter: exclude cancelled flights

Per ADR-0002, we exclude cancelled flights from all delay analysis.

In [ ]:
print(f'Total rows:     {len(df):,}')
print(f'Cancelled == 1: {(df["Cancelled"] == 1).sum():,}')
df_active = df[df['Cancelled'] == 0].copy()
print(f'Active flights: {len(df_active):,}')
print(f'\nArrDel15 missing in active flights: {df_active["ArrDel15"].isnull().sum()}')

## 4. Target variable distribution (`ArrDel15`)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
counts = df_active['ArrDel15'].value_counts().sort_index()
ax.bar(['On time / < 15 min', 'Delayed >= 15 min'], counts.values, color=['#4CAF50', '#F44336'])
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f'{v:,}\n({v/len(df_active)*100:.1f}%)', ha='center', fontsize=10)
ax.set_ylabel('Flights')
ax.set_title('Arrival Delay >= 15 min (active flights)')
plt.tight_layout()
plt.show()

## 5. Delay rate by Day of Week

In [ ]:
by_dow = df_active.groupby('DayOfWeek')['ArrDel15'].mean().reset_index()
by_dow['DayName'] = by_dow['DayOfWeek'].map(WEEKDAY_NAMES)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(by_dow['DayName'], by_dow['ArrDel15'] * 100, color='#5B8DEF')
ax.set_ylabel('Delay rate (%)')
ax.set_title('Arrival delay >= 15 min rate by day of week')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Delay rate by Destination Airport (top 20)

In [ ]:
by_dest = (
    df_active.groupby(['DestAirportID', 'DestAirportName'])['ArrDel15']
    .agg(['mean', 'count'])
    .reset_index()
)
by_dest.columns = ['DestAirportID', 'DestAirportName', 'delay_rate', 'flights']
by_dest_top = by_dest[by_dest['flights'] >= 100].nlargest(20, 'delay_rate')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(by_dest_top['DestAirportName'], by_dest_top['delay_rate'] * 100, color='#FF7043')
ax.set_xlabel('Delay rate (%)')
ax.set_title('Top 20 destination airports by delay rate (min 100 flights)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Delay rate by Carrier

In [ ]:
by_carrier = (
    df_active.groupby('Carrier')['ArrDel15']
    .agg(['mean', 'count'])
    .reset_index()
    .sort_values('mean', ascending=False)
)
by_carrier.columns = ['Carrier', 'delay_rate', 'flights']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(by_carrier['Carrier'], by_carrier['delay_rate'] * 100, color='#AB47BC')
ax.set_ylabel('Delay rate (%)')
ax.set_title('Arrival delay >= 15 min rate by carrier')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Delay rate by Month

In [ ]:
by_month = df_active.groupby('Month')['ArrDel15'].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(by_month['Month'], by_month['ArrDel15'] * 100, color='#26A69A')
ax.set_xticks(range(1, 13))
ax.set_xlabel('Month')
ax.set_ylabel('Delay rate (%)')
ax.set_title('Arrival delay >= 15 min rate by month')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 9. Correlation heatmap

In [ ]:
numeric_cols = ['Month', 'DayofMonth', 'DayOfWeek', 'CRSDepTime', 'DepDelay',
                'DepDel15', 'CRSArrTime', 'ArrDelay', 'ArrDel15']
corr = df_active[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation matrix (active flights)')
plt.tight_layout()
plt.show()

## 10. Key findings

_Fill in after reviewing the charts above:_

- **Overall delay rate**: ~__%
- **Worst day of week**: ___
- **Best day of week**: ___
- **Highest-delay carriers**: ___
- **Seasonal pattern**: ___
- **Strong correlations with ArrDel15**: ___